In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_KEY")
groq_api_key

'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'

In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="Gemma2-9b-It",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001712438B550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017124354A00>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

model.invoke([HumanMessage(content="Hi, my name is Muskan and I am data scientist")])

AIMessage(content="Hello Muskan, it's nice to meet you! \n\nIt's great to know you're a data scientist. \n\nWhat kind of work do you do in the field? Are you working on any interesting projects right now?  \n\nI'm always eager to learn more about the work that data scientists do.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 21, 'total_tokens': 92, 'completion_time': 0.129090909, 'prompt_time': 0.002273726, 'queue_time': 0.230582215, 'total_time': 0.131364635}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-066ac3d2-5376-409f-bed5-b64051ac2b3a-0', usage_metadata={'input_tokens': 21, 'output_tokens': 71, 'total_tokens': 92})

In [5]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="my name is Muskan and I am a data scientist"),
        AIMessage(content="Hello Muskan nice to meet you.\n as a data scientist what do you do"),
        HumanMessage(content="Hey whats my name and what you do"),

    ]

)

AIMessage(content="You said your name is Muskan, and you're a data scientist! \n\nAs for me, I am Gemma, an AI assistant. I can process and generate text, answer your questions, and help you with various language-based tasks. \n\nWhat can I help you with today, Muskan?\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 55, 'total_tokens': 122, 'completion_time': 0.121818182, 'prompt_time': 0.003437093, 'queue_time': 0.23343471999999998, 'total_time': 0.125255275}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-89733608-1e30-41ef-a7fb-8ca408fa148b-0', usage_metadata={'input_tokens': 55, 'output_tokens': 67, 'total_tokens': 122})

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)


In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi my name is muskan and I am a data scientist")],
    config=config
)

In [9]:
response.content

"Hello Muskan! \n\nIt's nice to meet you.  \n\nWhat kind of data science work do you do? \n\nI'm always interested in learning more about how people use AI and data analysis in their work.\n"

In [10]:
##change session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi what is my name")],
    config=config1
)
response.content

'As an AI, I have no memory of past conversations and do not know your name. Could you please tell me? 😊\n'

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi my name is gauri")],
    config=config1
)
response.content

"Hello Gauri, it's nice to meet you! 👋  Is there anything I can help you with today?  \n"

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config=config1
)
response.content

'Your name is Gauri.  😊  \n\nI remember!  How can I help you today, Gauri?  \n'

In [16]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(

    [
        ("system, you are helpful assistant answer all the question to best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain=prompt|model

In [17]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is Muskan")]})

AIMessage(content="Hello Muskan, it's nice to meet you!\n\nHow can I help you today? 😊  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 40, 'total_tokens': 65, 'completion_time': 0.045454545, 'prompt_time': 0.002366004, 'queue_time': 0.23163577, 'total_time': 0.047820549}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-95d4b3b1-44d3-4545-8722-384911278aec-0', usage_metadata={'input_tokens': 40, 'output_tokens': 25, 'total_tokens': 65})

In [18]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [19]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="My name is muskan")],
    config=config
)
response.content

"Hello Muskan, it's nice to meet you! 👋 \n\nIs there anything I can help you with today? 😊  \n"

In [20]:
response=with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config=config
)
response.content

'Your name is Muskan. 😊  \n\nI remember! \n\n\n'

In [21]:
##add more complexity

prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "you are helpful assistant. Answer all questions to best of quality in {language}"
        ),

        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain=prompt|model

In [31]:
config4={"configurable":{"session_id":"chat4"}}

response=chain.invoke({"messages":[HumanMessage(content="Hi my name is Muskan")],"language":"Hindi"},config=config4)
response.content

'नमस्ते मुस्कान!  😊 \n\nमैं आपकी मदद करने के लिए तैयार हूँ। आप मुझसे कोई भी प्रश्न पूछ सकते हैं, मैं अपना सर्वश्रेष्ठ प्रयास करूँगा कि आपको सटीक और उपयोगी उत्तर प्रदान कर सकूँ। \n\n'

In [33]:
response=chain.invoke({"messages":[HumanMessage(content="what is my name")],"language":"Hindi"},config=config4,)
response.content

'मुझे तुम्हारा नाम नहीं पता। मैं एक बड़ा भाषा मॉडल हूँ, मुझे लोगों के बारे में व्यक्तिगत जानकारी नहीं पता है। \n\nक्या मैं तुम्हें कुछ और मदद कर सकता हूँ? \n\n'

In [35]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\MUSKAN\data_science_muskan\Genai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\MUSKAN\data_science_muskan\Genai\venv\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MUSKAN\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [36]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As an AI, I don't have access to your personal preferences like your favorite ice cream flavor.  \n\nWhat's your favorite ice cream flavor?  🍦😊\n"

In [37]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As a helpful assistant, I don't have access to your personal information, including your taste preferences.  \n\nWhat's your favorite flavor of ice cream? 😊🍦\n"

In [38]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [39]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As an AI, I have no memory of past conversations and don't know your name.  What's your name? 😊\n"

In [40]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I have no memory of past conversations.  \n\nIf you'd like to ask me a math problem, I'm happy to help!  Just let me know what it is. 😊 \n\n"